# Preparación de datos (opcional)

Basado en `notebook_1.ipynb`. Requiere el CSV original de delitos (~2.21 GB),
no incluido. El análisis principal usa directamente los datos agregados incluidos.
El filtro implementado es `Year >= 2015`, con exclusión de coordenadas nulas.
Descargar una versión actualizada de la fuente no recupera exactamente el corte original.
La ejecución sobrescribe los CSV de `data/processed`; conserve sus versiones si desea comparar.


In [ ]:
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "stan_models/poisson_offset.stan").exists()
             and (p / "data/processed/final_data_2.0.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Abra el notebook desde el repositorio descargado.")
RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"
RESULTS_DIR = ROOT / "results/generated"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import polars as pl


In [ ]:
from pathlib import Path

csv_file = next(RAW_DIR.glob("Census_Data*_20260329.csv"))
socioecon_df = pd.read_csv(
    csv_file.resolve().as_posix(),
    encoding="utf-8-sig",
    engine="python"
)
display(socioecon_df.head())
socioecon_df.info()

In [ ]:
# Correct data types for socioecon_df
# Convert percentage columns from string with commas to float
percentage_cols = [
    'PERCENT OF HOUSING CROWDED',
    'PERCENT HOUSEHOLDS BELOW POVERTY',
    'PERCENT AGED 16+ UNEMPLOYED',
    'PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA',
    'PERCENT AGED UNDER 18 OR OVER 64'
]

for col in percentage_cols:
    socioecon_df[col] = socioecon_df[col].str.replace(',', '.').astype(float)

# Optionally, convert Community Area Number to int, but keep as float due to NaN
socioecon_df['Community Area Number'] = socioecon_df['Community Area Number'].astype('Int32')  # if using pandas nullable int

# Verify the changes
print(socioecon_df.dtypes)
socioecon_df.head(2)

In [ ]:
import polars as pl

crimes_df = pl.scan_csv(RAW_DIR / "Crimes_-_2001_to_Present_20260329.csv")

# ver primeras filas
display(crimes_df.collect().head())

# schema
print(crimes_df.collect_schema)

# número de filas (eficiente)
print(crimes_df.select(pl.len()).collect())

# columnas
crimes_df.collect_schema().names()

In [ ]:
crimes_df = (
    pl.scan_csv(RAW_DIR / "Crimes_-_2001_to_Present_20260329.csv")
    
    # 🎯 seleccionar solo lo importante
    .select([
        "Date",
        "Primary Type",
        "Description",
        "Arrest",
        "Domestic",
        "Latitude",
        "Longitude",
        "Community Area",
        "Year"
    ])
    
    # 🧹 limpiar datos basura
    .drop_nulls(["Latitude", "Longitude"])
)

crimes_df = crimes_df.with_columns(
    pl.col("Date").str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p")
)

crimes_df = crimes_df.with_columns([
    pl.col("Date").dt.weekday().alias("weekday"),
    pl.col("Date").dt.month().alias("month"),
     pl.col("Community Area").cast(pl.Int32)    # hago esto para que coincida con el tipo de dato en socioecon_df.
])


# ver primeras filas
display(crimes_df.collect().head())

# schema
print(crimes_df.collect_schema)

# número de filas (eficiente)
print(crimes_df.select(pl.len()).collect())

# columnas
crimes_df.collect_schema().names()

In [ ]:
# Obtener etiquetas únicas y sus frecuencias en "Primary Type"

primary_type_counts = crimes_df.group_by("Primary Type").agg(
    pl.len().alias("frequency")
).sort("frequency", descending=True).collect()

# Mostrar las etiquetas y sus frecuencias
# print("Etiquetas únicas en 'Primary Type' y sus frecuencias:")
# print(primary_type_counts)

# Crear diccionario con etiquetas como claves y frecuencias como valores
primary_type_dict = primary_type_counts.to_pandas().set_index('Primary Type')['frequency'].to_dict()

print("\nDiccionario de frecuencias:")
primary_type_dict

In [ ]:
import polars as pl

# =========================
# 1. FILTRAR DATOS
# =========================

df_filtered = crimes_df.filter(
    (pl.col("Year") >= 2015)
)

# =========================
# 2. DEFINIR CATEGORÍAS
# =========================

violent = [
    "BATTERY", "ASSAULT", "ROBBERY", "HOMICIDE",
    "CRIMINAL SEXUAL ASSAULT", "CRIM SEXUAL ASSAULT",
    "WEAPONS VIOLATION", "KIDNAPPING"
]

property_ = [
    "THEFT", "BURGLARY", "MOTOR VEHICLE THEFT",
    "CRIMINAL DAMAGE", "ARSON"
]

# =========================
# 3. CREAR COLUMNA CATEGÓRICA
# =========================

df_cat = df_filtered.with_columns(
    pl.when(pl.col("Primary Type").is_in(violent))
    .then(pl.lit("violent"))
    .when(pl.col("Primary Type").is_in(property_))
    .then(pl.lit("property"))
    .otherwise(pl.lit("other"))
    .alias("crime_category")
)

# =========================
# 4. AGREGAR POR ÁREA Y CATEGORÍA
# =========================

agg_cat = df_cat.group_by(
    ["Community Area", "crime_category"]
).agg(
    pl.len().alias("count")
).collect()

# =========================
# 5. PIVOTEAR
# =========================

pivot_cat = agg_cat.pivot(
    values="count",
    index="Community Area",
    on="crime_category"
).fill_null(0)

# =========================
# 6. TOTAL CRÍMENES
# =========================

pivot_cat = pivot_cat.with_columns(
    (pl.col("violent") + pl.col("property") + pl.col("other"))
    .alias("total_crimes")
)

# =========================
# 7. PASAR A PANDAS
# =========================

agg_df = pivot_cat.to_pandas()

# =========================
# 8. MERGE CON SOCIOECONÓMICO
# =========================

final_df = agg_df.merge(
    socioecon_df,
    left_on="Community Area",
    right_on="Community Area Number"
)

# =========================
# 9. LIMPIEZA FINAL
# =========================

# ordenar
final_df = final_df.sort_values("Community Area Number").reset_index(drop=True)

# eliminar columna duplicada
final_df.drop(columns=["Community Area"], inplace=True)

# eliminar NA (solo 1 fila)
final_df.dropna(inplace=True)

# cambiar nombre de la columna "PER CAPITA INCOME " (con espacio al final) a "PER CAPITA INCOME"
final_df.rename(columns={"PER CAPITA INCOME ": "PER CAPITA INCOME"}, inplace=True)

# en la columna "COMMUNITY AREA NAME" hay un error de tipeo en el área 18, se llama "Montclaire" pero debería ser "Montclare". Corregimos eso:
final_df["COMMUNITY AREA NAME"] = final_df["COMMUNITY AREA NAME"].replace("Montclaire", "Montclare")

# =========================
# 10. FEATURES EXTRAS (MUY RECOMENDADO)
# =========================

final_df["pct_violent"] = final_df["violent"] / final_df["total_crimes"]    # la más importante, el porcentaje de crímenes violentos es un indicador clave de la seguridad en un área.
final_df["pct_property"] = final_df["property"] / final_df["total_crimes"]
final_df["pct_other"] = final_df["other"] / final_df["total_crimes"]

# =========================
# 11. Ajustar orden de columnas
# =========================

final_df = final_df[[
    "Community Area Number",
    "COMMUNITY AREA NAME",
    "total_crimes",
    "violent",
    "pct_violent",
    "property",
    "pct_property",
    "other",
    "pct_other",
    "PERCENT OF HOUSING CROWDED",
    "PERCENT HOUSEHOLDS BELOW POVERTY",
    "PERCENT AGED 16+ UNEMPLOYED",
    "PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA",
    "PERCENT AGED UNDER 18 OR OVER 64",
    "PER CAPITA INCOME",
    "HARDSHIP INDEX"
]]

# =========================
# 12. RESULTADO FINAL
# =========================

display(final_df)
final_df.info()

In [ ]:
## agregar conteo de arrestos por área comunitaria, discriminando por tipo de crimen (violent, property, other) para luego calcular el porcentaje de arrestos en cada categoría. Esto nos permitirá analizar si hay diferencias significativas en las tasas de arresto entre los diferentes tipos de crímenes y cómo se relacionan con las características socioeconómicas de cada área comunitaria.

arrest_pivot = (
    df_cat
    .with_columns(
        pl.col("Arrest").cast(pl.Boolean, strict=False).fill_null(False).alias("Arrest")
    )
    .group_by(["Community Area", "crime_category"])
    .agg(pl.col("Arrest").sum().alias("arrests"))
    .collect()
    .pivot(values="arrests", index="Community Area", on="crime_category")
    .fill_null(0)
)

# asegurar columnas aunque alguna categoría no aparezca
for c in ["violent", "property", "other"]:
    if c not in arrest_pivot.columns:
        arrest_pivot = arrest_pivot.with_columns(pl.lit(0).alias(c))

arrest_pivot = arrest_pivot.rename({
    "violent": "violent_arrests",
    "property": "property_arrests",
    "other": "other_arrests"
}).with_columns(
    (
        pl.col("violent_arrests")
        + pl.col("property_arrests")
        + pl.col("other_arrests")
    ).alias("total_arrests")
)

arrests_df = arrest_pivot.to_pandas()

# permitir re-ejecutar la celda sin duplicar columnas
cols_to_drop = [
    "Community Area",
    "violent_arrests", "property_arrests", "other_arrests", "total_arrests",
    "pct_arrests_violent", "pct_arrests_property", "pct_arrests_other",
    "arrest_rate_violent", "arrest_rate_property", "arrest_rate_other", "arrest_rate_total"
]
final_df = final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns], errors="ignore")

# merge con la tabla final
final_df = final_df.merge(
    arrests_df,
    left_on="Community Area Number",
    right_on="Community Area",
    how="left"
).drop(columns=["Community Area"])

arrest_cols = ["violent_arrests", "property_arrests", "other_arrests", "total_arrests"]
final_df[arrest_cols] = final_df[arrest_cols].fillna(0).astype("uint32")


# tasa de arresto por categoría (arrestos / delitos de esa categoría)
final_df["arrest_rate_violent"] = np.where(
    final_df["violent"] > 0,
    final_df["violent_arrests"] / final_df["violent"],
    np.nan
)
final_df["arrest_rate_property"] = np.where(
    final_df["property"] > 0,
    final_df["property_arrests"] / final_df["property"],
    np.nan
)
final_df["arrest_rate_other"] = np.where(
    final_df["other"] > 0,
    final_df["other_arrests"] / final_df["other"],
    np.nan
)
final_df["arrest_rate_total"] = np.where(
    final_df["total_crimes"] > 0,
    final_df["total_arrests"] / final_df["total_crimes"],
    np.nan
)

# reordenar columnas
final_df = final_df[
    [
        "Community Area Number",
        "COMMUNITY AREA NAME",
        "total_crimes",
        "total_arrests",
        "arrest_rate_total",
        "violent",
        "violent_arrests",
        "pct_violent",
        "arrest_rate_violent",
        "property",
        "property_arrests",
        "pct_property",
        "arrest_rate_property",
        "other",
        "other_arrests",
        "pct_other",
        "arrest_rate_other",
        "PERCENT OF HOUSING CROWDED",
        "PERCENT HOUSEHOLDS BELOW POVERTY",
        "PERCENT AGED 16+ UNEMPLOYED",
        "PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA",
        "PERCENT AGED UNDER 18 OR OVER 64",
        "PER CAPITA INCOME",
        "HARDSHIP INDEX"
    ]
]

display(final_df)
final_df.info()

In [ ]:
## guardar resultado final
# guardar en la misma carpeta del archivo de entrada
output_file = PROCESSED_DIR / "final_data.csv"
final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Archivo guardado en: {output_file}")    

In [ ]:
from pathlib import Path

# Busca el CSV en el directorio actual (o ajusta la ruta si está en otro lugar)
csv_path = next(RAW_DIR.glob("ACS_5_Year_Data_by_Community_Area_20260330*.csv"), None)
if csv_path is None:
    raise FileNotFoundError("No se encontró ACS_5_Year_Data_by_Community_Area_20260330.csv en el directorio actual")

census_df = pd.read_csv(csv_path.resolve().as_posix(), encoding="utf-8-sig", engine="python")

display(census_df)
census_df.info()

In [ ]:
pop_df = census_df[["Community Area", "Total Population"]]
pop_df

In [ ]:
def normalizar_nombre_area(serie):
    return (
        serie.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"[^A-Z0-9 ]", "", regex=True)
        .replace({"WASHINGTON HEIGHT": "WASHINGTON HEIGHTS"})
    )

pop_df_clean = pop_df.copy()
pop_df_clean["merge_key"] = normalizar_nombre_area(pop_df_clean["Community Area"])
pop_df_clean["Total Population"] = (
    pop_df_clean["Total Population"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

final_df_with_popu = (
    final_df.assign(
        merge_key=normalizar_nombre_area(final_df["COMMUNITY AREA NAME"])
    )
    .merge(
        pop_df_clean[["merge_key", "Total Population"]],
        on="merge_key",
        how="left"
    )
    .drop(columns=["merge_key"])
)

final_df_with_popu["crime_rate"] = final_df_with_popu["total_crimes"] / final_df_with_popu["Total Population"]
final_df_with_popu["violent_rate"] = final_df_with_popu["violent"] / final_df_with_popu["Total Population"]
final_df_with_popu["property_rate"] = final_df_with_popu["property"] / final_df_with_popu["Total Population"]
final_df_with_popu["other_rate"] = final_df_with_popu["other"] / final_df_with_popu["Total Population"]

# cambiar tipo de dato de "Total Population" a int para evitar problemas en análisis posteriores
final_df_with_popu["Total Population"] = (
    pd.to_numeric(final_df_with_popu["Total Population"], errors="coerce")
    .astype("Int32")
)

final_df_with_popu = final_df_with_popu[
    ["Community Area Number",
    "COMMUNITY AREA NAME",
    "Total Population",
    "total_crimes",
    "crime_rate",
    "total_arrests",
    "arrest_rate_total",
    "violent",
    "pct_violent",
    "violent_rate",
    "violent_arrests",
    "arrest_rate_violent",
    "property",
    "pct_property",
    "property_rate",
    "property_arrests",
    "arrest_rate_property",
    "other",
    "pct_other",
    "other_rate",
    "other_arrests",
    "arrest_rate_other",
    "PERCENT OF HOUSING CROWDED",
    "PERCENT HOUSEHOLDS BELOW POVERTY",
    "PERCENT AGED 16+ UNEMPLOYED",
    "PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA",
    "PERCENT AGED UNDER 18 OR OVER 64",
    "PER CAPITA INCOME",
    "HARDSHIP INDEX"]]

display(final_df_with_popu)
#print(final_df_with_popu.shape)
print(final_df_with_popu.info())
print("Filas sin población:", final_df_with_popu["Total Population"].isna().sum())
print("Áreas comunitarias sin población:")
print(final_df_with_popu[final_df_with_popu["Total Population"].isna()]["COMMUNITY AREA NAME"].tolist())

In [ ]:
## guardar resultado final (2.0, con población incluida)
# guardar en la misma carpeta del archivo de entrada
output_file = PROCESSED_DIR / "final_data_2.0.csv"
final_df_with_popu.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Archivo guardado en: {output_file}") 